In [ ]:
# ============================================================
# 따릉이 최종 데이터셋 XGBoost 학습 코드
#
# 사용 데이터 위치:
#   C:\Users\Playdata\Desktop\final_model_data-20260501T051943Z-3-001\final_model_data
#
# 규칙:
#   1. FINAL_FEATURE_COLS 수정 금지
#   2. 데이터 전처리/피처 생성 추가 금지
#   3. 모델은 XGBoost 사용
#   4. 평가 지표는 MAE, RMSE, WAPE, R2 공통 사용
#   5. 모델은 .pkl로 저장
# ============================================================

import gc
import json
import pickle
import warnings
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")


# ============================================================
# 0. 기본 설정
# ============================================================

FINAL_DATA_DIR = Path(r"C:\Users\Playdata\Desktop\final_model_data-20260501T051943Z-3-001\final_model_data")

TRAIN_DIR = FINAL_DATA_DIR / "train_2023_parts"
VALID_DIR = FINAL_DATA_DIR / "valid_2024_parts"
TEST_DIR = FINAL_DATA_DIR / "test_2025_parts"

MODEL_DIR = FINAL_DATA_DIR / "data" / "model_3"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "rental_count"
RANDOM_SEED = 42


# ============================================================
# 1. 실험 설정
# ============================================================

MODEL_VERSION = "xgboost_tuned_v1"
MODEL_TYPE = "xgboost"

OVERWRITE = False
SAVE_FEATURE_IMPORTANCE = True

SAVE_MODEL_PATH = MODEL_DIR / f"{MODEL_VERSION}.pkl"
SAVE_METRICS_PATH = MODEL_DIR / f"{MODEL_VERSION}_metrics.csv"
SAVE_IMPORTANCE_PATH = MODEL_DIR / f"{MODEL_VERSION}_feature_importance.csv"
SAVE_CONFIG_PATH = MODEL_DIR / f"{MODEL_VERSION}_config.json"


# ============================================================
# 2. 최종 고정 피처
# ============================================================

FINAL_FEATURE_COLS = [
    "station_id",
    "latitude",
    "longitude",
    "rack_count",
    "station_age_days",

    "month",
    "day",
    "hour",
    "dayofweek",
    "is_day_off",

    "hour_sin",
    "hour_cos",

    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",

    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",

    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    "rolling_std_24",

    "diff_1",
    "diff_24",
]


# ============================================================
# 3. XGBoost 하이퍼파라미터
# ============================================================

XGBOOST_PARAMS = {
    "objective": "count:poisson",
    "eval_metric": "rmse",

    # 기존 0.05보다 낮춰서 더 천천히 학습
    # 대용량 시계열 데이터에서는 과적합을 줄이고 일반화 성능을 올리는 방향
    "learning_rate": 0.03,

    # 기존 8보다 깊게 해서 시간/대여소/lag 간 상호작용을 더 잘 잡도록 조정
    "max_depth": 10,

    # 기존 10보다 크게 해서 너무 작은 패턴에 과적합되는 것을 억제
    "min_child_weight": 30,

    # 샘플링을 약간 낮춰서 일반화 성능 개선 시도
    "subsample": 0.80,
    "colsample_bytree": 0.80,

    # 보수적 정규화 강화
    "reg_alpha": 0.3,
    "reg_lambda": 2.0,

    # 분할 손실 감소 조건 추가
    # 의미 없는 작은 분할을 줄여 오차 흔들림을 완화
    "gamma": 0.1,

    "tree_method": "hist",
    "random_state": RANDOM_SEED,

    # learning_rate를 낮춘 만큼 트리 수 증가
    "n_estimators": 3000,

    "n_jobs": -1,
}


# ============================================================
# 4. 시간 출력 유틸
# ============================================================

def format_seconds(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}시간 {m}분 {s}초"
    if m > 0:
        return f"{m}분 {s}초"
    return f"{s}초"


def print_elapsed(title: str, start_time: float):
    elapsed = time.perf_counter() - start_time
    print(f"[소요시간] {title}: {format_seconds(elapsed)}")


def check_output_paths():
    if OVERWRITE:
        return

    existing = []

    for path in [SAVE_MODEL_PATH, SAVE_METRICS_PATH, SAVE_IMPORTANCE_PATH, SAVE_CONFIG_PATH]:
        if path.exists():
            existing.append(path)

    if existing:
        print("[중복 파일 발견]")
        for p in existing:
            print(" -", p)

        raise FileExistsError(
            "같은 MODEL_VERSION 결과 파일이 이미 있습니다. "
            "MODEL_VERSION을 바꾸거나 OVERWRITE=True로 설정하세요."
        )


# ============================================================
# 5. 데이터 로드 함수
# ============================================================

def check_data_paths():
    print("TRAIN_DIR:", TRAIN_DIR)
    print("VALID_DIR:", VALID_DIR)
    print("TEST_DIR :", TEST_DIR)

    train_count = len(list(TRAIN_DIR.glob("*.parquet")))
    valid_count = len(list(VALID_DIR.glob("*.parquet")))
    test_count = len(list(TEST_DIR.glob("*.parquet")))

    print("train parquet 개수:", train_count)
    print("valid parquet 개수:", valid_count)
    print("test parquet 개수 :", test_count)

    if train_count == 0:
        raise FileNotFoundError(f"train parquet 파일이 없습니다: {TRAIN_DIR}")
    if valid_count == 0:
        raise FileNotFoundError(f"valid parquet 파일이 없습니다: {VALID_DIR}")
    if test_count == 0:
        raise FileNotFoundError(f"test parquet 파일이 없습니다: {TEST_DIR}")


def load_parquet_parts(folder: Path, columns=None, name="data") -> pd.DataFrame:
    files = sorted(folder.glob("*.parquet"))

    if len(files) == 0:
        raise FileNotFoundError(f"{folder} 안에 parquet 파일이 없습니다.")

    start = time.perf_counter()
    dfs = []

    for p in tqdm(files, desc=f"{name} 로드 중"):
        part = pd.read_parquet(p, columns=columns)
        dfs.append(part)

    df = pd.concat(dfs, ignore_index=True)

    del dfs
    gc.collect()

    print(f"{name} shape:", df.shape)
    print_elapsed(f"{name} 로드", start)

    return df


def validate_columns(df: pd.DataFrame, name: str):
    required_cols = FINAL_FEATURE_COLS + [TARGET]
    missing_cols = [c for c in required_cols if c not in df.columns]

    if missing_cols:
        raise ValueError(f"{name} 데이터에 필요한 컬럼이 없습니다: {missing_cols}")

    target_na = int(df[TARGET].isna().sum())
    if target_na > 0:
        raise ValueError(f"{name}: {TARGET} 결측치가 있습니다: {target_na}")

    negative_count = int((df[TARGET] < 0).sum())
    if negative_count > 0:
        raise ValueError(f"{name}: {TARGET} 음수 값이 있습니다: {negative_count}")

    return True


# ============================================================
# 6. 평가 함수
# ============================================================

def regression_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    y_pred = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    denominator = np.sum(np.abs(y_true))
    wape = np.sum(np.abs(y_true - y_pred)) / denominator if denominator != 0 else np.nan

    r2 = r2_score(y_true, y_pred)

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "WAPE": float(wape),
        "R2": float(r2),
    }


def print_metrics(title: str, metrics: dict):
    print("=" * 80)
    print(title)
    print("=" * 80)
    print(f"MAE  : {metrics['MAE']:.6f}")
    print(f"RMSE : {metrics['RMSE']:.6f}")
    print(f"WAPE : {metrics['WAPE']:.6f}")
    print(f"R2   : {metrics['R2']:.6f}")
    print("=" * 80)


# ============================================================
# 7. XGBoost 학습 함수
# ============================================================

def train_xgboost(X_train, y_train, X_valid, y_valid):
    from xgboost import XGBRegressor

    start = time.perf_counter()

    model = XGBRegressor(**XGBOOST_PARAMS)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=100,
    )

    print_elapsed("XGBoost 학습", start)

    return model


# ============================================================
# 8. 예측 및 저장 유틸
# ============================================================

def predict_model(model, X):
    pred = model.predict(X)
    pred = np.clip(pred, 0, None)
    return pred


def get_model_params_for_save():
    return {
        "params": XGBOOST_PARAMS,
    }


def save_xgboost_importance(model):
    if not SAVE_FEATURE_IMPORTANCE:
        return

    if not hasattr(model, "feature_importances_"):
        return

    importance_df = pd.DataFrame({
        "feature": FINAL_FEATURE_COLS,
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    importance_df.to_csv(SAVE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")
    print("feature importance 저장:", SAVE_IMPORTANCE_PATH)


# ============================================================
# 9. 실행
# ============================================================

def main():
    total_start = time.perf_counter()

    print("=" * 100)
    print("[XGBoost 학습 코드 시작]")
    print("=" * 100)

    print("MODEL_VERSION:", MODEL_VERSION)
    print("MODEL_TYPE:", MODEL_TYPE)
    print("MODEL_DIR:", MODEL_DIR)

    check_data_paths()
    check_output_paths()

    # --------------------------------------------------------
    # 데이터 로드
    # --------------------------------------------------------
    stage_start = time.perf_counter()

    load_cols = FINAL_FEATURE_COLS + [TARGET]

    train = load_parquet_parts(TRAIN_DIR, columns=load_cols, name="train_2023")
    valid = load_parquet_parts(VALID_DIR, columns=load_cols, name="valid_2024")
    test = load_parquet_parts(TEST_DIR, columns=load_cols, name="test_2025")

    validate_columns(train, "train")
    validate_columns(valid, "valid")
    validate_columns(test, "test")

    print_elapsed("전체 데이터 로드 및 검증", stage_start)

    # --------------------------------------------------------
    # X / y 생성
    # --------------------------------------------------------
    stage_start = time.perf_counter()

    X_train = train[FINAL_FEATURE_COLS].copy()
    y_train = train[TARGET].copy()

    X_valid = valid[FINAL_FEATURE_COLS].copy()
    y_valid = valid[TARGET].copy()

    X_test = test[FINAL_FEATURE_COLS].copy()
    y_test = test[TARGET].copy()

    print("\nX_train:", X_train.shape)
    print("X_valid:", X_valid.shape)
    print("X_test :", X_test.shape)
    print("사용 피처 수:", len(FINAL_FEATURE_COLS))

    del train, valid, test
    gc.collect()

    print_elapsed("X/y 생성", stage_start)

    # --------------------------------------------------------
    # 모델 학습
    # --------------------------------------------------------
    print("\n" + "=" * 100)
    print("[XGBoost 모델 학습]")
    print("=" * 100)

    model = train_xgboost(X_train, y_train, X_valid, y_valid)

    # --------------------------------------------------------
    # 예측 및 평가
    # --------------------------------------------------------
    stage_start = time.perf_counter()

    print("\n" + "=" * 100)
    print("[예측 및 평가]")
    print("=" * 100)

    pred_valid = predict_model(model, X_valid)
    pred_test = predict_model(model, X_test)

    valid_metrics = regression_metrics(y_valid, pred_valid)
    test_metrics = regression_metrics(y_test, pred_test)

    print_metrics("Validation 성능", valid_metrics)
    print_metrics("Test 성능", test_metrics)

    print_elapsed("예측 및 평가", stage_start)

    # --------------------------------------------------------
    # Metrics 저장
    # --------------------------------------------------------
    metrics_df = pd.DataFrame([
        {
            "model_version": MODEL_VERSION,
            "model_type": MODEL_TYPE,
            "dataset": "valid_2024",
            **valid_metrics,
            "n_rows": int(len(y_valid)),
            "target_sum": float(y_valid.sum()),
            "target_mean": float(y_valid.mean()),
            "zero_ratio": float((y_valid == 0).mean()),
            "feature_count": int(len(FINAL_FEATURE_COLS)),
        },
        {
            "model_version": MODEL_VERSION,
            "model_type": MODEL_TYPE,
            "dataset": "test_2025",
            **test_metrics,
            "n_rows": int(len(y_test)),
            "target_sum": float(y_test.sum()),
            "target_mean": float(y_test.mean()),
            "zero_ratio": float((y_test == 0).mean()),
            "feature_count": int(len(FINAL_FEATURE_COLS)),
        },
    ])

    print("\n[Metrics]")
    print(metrics_df)

    metrics_df.to_csv(SAVE_METRICS_PATH, index=False, encoding="utf-8-sig")
    print("\nmetrics 저장:", SAVE_METRICS_PATH)

    # --------------------------------------------------------
    # Feature importance 저장
    # --------------------------------------------------------
    save_xgboost_importance(model)

    # --------------------------------------------------------
    # Config 저장
    # --------------------------------------------------------
    config = {
        "model_version": MODEL_VERSION,
        "model_type": MODEL_TYPE,
        "target": TARGET,
        "random_seed": RANDOM_SEED,
        "feature_count": len(FINAL_FEATURE_COLS),
        "feature_cols": FINAL_FEATURE_COLS,
        "model_params": get_model_params_for_save(),
        "train_dir": str(TRAIN_DIR),
        "valid_dir": str(VALID_DIR),
        "test_dir": str(TEST_DIR),
    }

    with open(SAVE_CONFIG_PATH, "w", encoding="utf-8") as f:
        json.dump(config, f, ensure_ascii=False, indent=2)

    print("config 저장:", SAVE_CONFIG_PATH)

    # --------------------------------------------------------
    # 모델 저장
    # --------------------------------------------------------
    save_bundle = {
        "model": model,
        "model_version": MODEL_VERSION,
        "model_type": MODEL_TYPE,
        "feature_cols": FINAL_FEATURE_COLS,
        "target_col": TARGET,
        "valid_metrics": valid_metrics,
        "test_metrics": test_metrics,
        "random_seed": RANDOM_SEED,
        "model_params": get_model_params_for_save(),
        "prediction_postprocess": "clip_negative_to_zero",
    }

    with open(SAVE_MODEL_PATH, "wb") as f:
        pickle.dump(save_bundle, f)

    print("model 저장:", SAVE_MODEL_PATH)

    print("\n" + "=" * 100)
    print("[XGBoost 학습 코드 완료]")
    print("전체 소요시간:", format_seconds(time.perf_counter() - total_start))
    print("=" * 100)


if __name__ == "__main__":
    main()
